# 07 · Bring your own model

Takes an arbitrary PyTorch model from `nn.Module` to a wired-in Brain-Score candidate in
four steps: **inspect** it to detect the appropriate wrapper and a provisional
region-to-layer map, **wrap** it, **extract** features at a mapped layer, and emit a
**registration scaffold** suitable for `brainscore/models/`.

The demonstration uses an architecture with random weights, so nothing is downloaded,
and a small set of synthetic images. Both are intended to be replaced.

## 1 · The model

Any `nn.Module`. A ResNet-18 with random weights is used here.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import torchvision.models as tvm

# Any PyTorch model works. This one has random weights, so it downloads nothing and
# predicts nothing useful -- fine, because this notebook is about the WIRING, not scores.
model = tvm.resnet18(weights=None).eval()   # <-- substitute the model under study
n_params = sum(p.numel() for p in model.parameters())   # numel = number of values in a tensor
print(f'{model.__class__.__name__}: {n_params:,} parameters')

ResNet: 11,689,512 parameters


## 2 · Inspection

`inspect_model` reads the architecture and returns a recommended wrapper together with a
*provisional* region-to-layer map. The map is a starting point rather than a committed
mapping; it is refined with the layer-mapping explorer against a real benchmark.

The region-to-layer map is a structural heuristic determining which layer is extracted
when V1, V4 or IT is requested. It is not evidence of brain correspondence, which is
established empirically.

In [2]:
from brainscore.tools.auto_register import inspect_model

# inspect_model reads the model's structure and suggests two things: which wrapper to
# use, and a first guess at which layer stands in for which brain region.
profile = inspect_model(model, identifier='my-resnet18')
print(profile.summary())
print('\nprovisional region_layer_map:')
# The map is provisional: a starting point rather than a finding. It is confirmed by
# scoring candidate layers against measured data.
for region, layer in profile.provisional_region_layer_map().items():
    print(f'  {region:3s} -> {layer}')

Model: my-resnet18
  parameters: 11,689,512  dtype: float32
  multimodal: False
  tower 0: vision  → VisionWrapper
    wrap: (full model)
    8 candidate layers (layer1.0 … layer4.1)
    why: vision backbone signal in 'ResNet'
    provisional map: V1=layer1.0, V2=layer2.0, V4=layer3.1, IT=layer4.1

provisional region_layer_map:
  V1  -> layer1.0
  V2  -> layer2.0
  V4  -> layer3.1
  IT  -> layer4.1


## 3 · Wrapping and feature extraction

`inspect_model` recommends `VisionWrapper`, which is used below. `VisionWrapper` is the
entry point for vision models: it inspects the model and dispatches internally to
`PytorchWrapper` for standard image models, `VLMVisionWrapper` for vision-language models
whose patches arrive concatenated rather than stacked, or `VideoWrapper` for
native-temporal models. One name therefore covers all three cases.

Several notebooks name the concrete classes directly. Those are the same components one
level below the facade, used where the individual stages are the subject of the
demonstration. For registration, `VisionWrapper` is the shorter path.

Synthetic images are used here; a real stimulus set is supplied the same way.

In [3]:
import tempfile, os, numpy as np
from PIL import Image
from brainscore.model_helpers.vision_wrapper import VisionWrapper
from brainscore_vision.model_helpers.activations.pytorch import load_preprocess_images

# Four random 64x64 images, standing in for a real stimulus set.
d = tempfile.mkdtemp()
paths = []
for i in range(4):
    p = os.path.join(d, f'img{i}.png')
    Image.fromarray((np.random.RandomState(i).rand(64, 64, 3) * 255).astype('uint8')).save(p)
    paths.append(p)

# VisionWrapper is the entry point for vision models: it inspects the model and selects
# the appropriate concrete wrapper internally, here PytorchWrapper.
it_layer = profile.provisional_region_layer_map()['IT']
wrapper = VisionWrapper(model, identifier='my-resnet18',
                        preprocessing=lambda imgs: load_preprocess_images(imgs, image_size=64))

# Run the images through and keep the values from the IT layer.
activations = wrapper(paths, layers=[it_layer])
print(f'features at IT layer ({it_layer}): {activations.shape}  '
      f'({activations.sizes["presentation"]} images x {activations.sizes["neuroid"]} units)')

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

features at IT layer (layer4.1): (4, 2048)  (4 images x 2048 units)


## 4 · The registration scaffold

`scaffold_registration` emits a starting-point plugin. Completing it requires supplying
the model loader and preprocessing, refining `REGION_LAYER_MAP` with the layer-mapping
explorer, placing the result in `brainscore/models/<name>/`, and scoring through the
`load_model` -> `load_benchmark` -> `score` path.

The scaffold terminates in `NotImplementedError` until the loader is supplied. The
verified result in this notebook is the feature extraction above: `(4, 2048)` activations
at the mapped layer.

In [4]:
from brainscore.tools.auto_register import scaffold_registration

# Print a starting-point registration file: the imports, the wrapper, and the region map,
# already filled in from what inspect_model found. Copy it into brainscore/models/<name>/
# and substitute the model loader.
print(scaffold_registration(profile))

"""Auto-generated registration scaffold for 'my-resnet18'.

Generated by brainscore.tools.auto_register. Fill in the model loader +
preprocessing, then refine REGION_LAYER_MAP with the layer-mapping explorer:

    from brainscore.tools.layer_mapping import sweep_model
    result, approaches, selector = sweep_model(
        wrapper, stimulus_set, brain_target, brain_stimulus_ids,
        layers=['layer1.0', 'layer1.1', 'layer2.0', 'layer2.1', 'layer3.0', 'layer3.1', 'layer4.0', 'layer4.1'])
    print(result.best_layer)   # → assign to the region you are mapping
"""
from brainscore import model_registry
from brainscore_core.model_interface import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper

# Provisional map (evenly spaced over detected blocks — REFINE with the
# layer-mapping explorer; this is only a runnable starting point).
REGION_LAYER_MAP = {
    'V1': 'layer1.0',
    'V2': 'layer2.0',
    'V4': 'layer3.1',
    'IT': 'layer4.1',
}


def get_mode

**Next steps.** Refine the mapping with `brainscore.tools.layer_mapping` against a real
benchmark, register the model, then call
`brainscore.score('my-resnet18', '<benchmark>')`. See
[EXTENDING.md](../EXTENDING.md).